# Body Tracking — Preprocessing Pipeline

**Output:** one `<participant_id>_cleaned_BT.csv` per participant, saved to `data/body_tracking/processed/`

### Pipeline overview
1. **Discover** participant IDs from raw CSV filenames
2. **Load & concatenate** all condition files (0, 1, 2, 3) for one participant
3. **Estimate sampling rate** from `raw_timestamp` (milliseconds)
4. **Add relative time axis** (`time_ms` since first sample of each model)
5. **Mark invalid samples** per tracker: `(x, y, z) == (0, 0, 0)` or non-finite
6. **Interpolate** expand bad-sample windows ±2 frames, linear interp for bracketed gaps
7. **Save** one CSV per participant

## Data structure

```
project/
└── data/
    └── body_tracking/
        ├── raw/
        │   ├── 001_BT_Data_Condition0_2026-04-29.csv
        │   ├── 001_BT_Data_Condition1_2026-04-29.csv
        │   ├── 001_BT_Data_Condition2_2026-04-29.csv
        │   ├── 001_BT_Data_Condition3_2026-04-29.csv
        │   ├── 002_BT_Data_Condition0_2026-04-29.csv
        │   └── ...
        └── processed/
            ├── 001_cleaned_BT.csv
            ├── 002_cleaned_BT.csv
            └── ...
```


# 1. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sns.set_style("whitegrid")

# Paths
DATA_DIR   = Path("../data/body_tracking/raw")
OUTPUT_DIR = Path("../data/body_tracking/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# File layout
# Pattern: 001_BT_Data_Condition0_2026-04-29.csv  (conditions 0–3 per participant)
CONDITIONS = [0, 1, 2, 3]

TRACKERS = ["RightFoot", "LeftFoot", "Waist", "LeftHand", "RightHand"]

POS_COLS = {T: [f"{T}_pos_x", f"{T}_pos_y", f"{T}_pos_z"] for T in TRACKERS}
ROT_COLS = {T: [f"{T}_rot_x", f"{T}_rot_y", f"{T}_rot_z", f"{T}_rot_w"] for T in TRACKERS}

# Progress-bar format
B_FORMAT = (
    "📄 {n_fmt} of {total_fmt} {desc} processed: {bar}\n"
    "    {percentage:3.0f}%  ⏱️ {elapsed}  ⏳ {remaining}  ⚙️ {rate_fmt}{postfix}"
)

print("Working directory:", Path.cwd())
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
print()
print("All CSV files found:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(" ", f.name)


Working directory: /Users/azad/Desktop/HiWi/lego-vr-analysis/eye-classification/notebooks
DATA_DIR: ../data/body_tracking/raw
DATA_DIR exists: True

All CSV files found:
  001_BT_Data_Condition0_2026-05-06.csv
  001_BT_Data_Condition1_2026-05-06.csv
  001_BT_Data_Condition2_2026-05-06.csv
  001_BT_Data_Condition3_2026-05-06.csv
  002_BT_Data_Condition0_2026-05-06.csv
  002_BT_Data_Condition1_2026-05-06.csv
  002_BT_Data_Condition2_2026-05-06.csv
  002_BT_Data_Condition3_2026-05-06.csv
  003_BT_Data_Condition0_2026-05-06.csv
  003_BT_Data_Condition1_2026-05-06.csv
  003_BT_Data_Condition2_2026-05-06.csv
  003_BT_Data_Condition3_2026-05-06.csv
  004_BT_Data_Condition0_2026-05-07.csv
  004_BT_Data_Condition1_2026-05-07.csv
  004_BT_Data_Condition2_2026-05-07.csv
  004_BT_Data_Condition3_2026-05-07.csv
  005_BT_Data_Condition0_2026-05-07.csv
  005_BT_Data_Condition1_2026-05-07.csv
  005_BT_Data_Condition2_2026-05-07.csv
  005_BT_Data_Condition3_2026-05-07.csv


# 2. Helper Functions

## 2.1 File Discovery & Loading

In [2]:
def get_participant_ids(data_dir=DATA_DIR):
    """Scan DATA_DIR for CSVs matching <id>_BT_Data_Condition<N>_*.csv."""
    csv_files = sorted(data_dir.glob("*_BT_Data_Condition*_*.csv"))
    return sorted({fp.name.split("_")[0] for fp in csv_files})


def get_participant_files(participant_id, data_dir=DATA_DIR, conditions=CONDITIONS):
    """Return one file path per condition for a given participant."""
    files = []
    for condition in conditions:
        matches = sorted(data_dir.glob(f"{participant_id}_BT_Data_Condition{condition}_*.csv"))
        if matches:
            files.append(matches[0])
    return files


def load_participant(file_paths, participant_id):
    """Load and concatenate all condition files for one participant."""
    dfs = []
    for fp in file_paths:
        d = pd.read_csv(fp, low_memory=False)
        d["source_file"] = fp.name
        dfs.append(d)
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df["participant_id"] = participant_id
    return df


participant_ids = get_participant_ids()
print("Participants found:", participant_ids)


Participants found: ['001', '002', '003', '004', '005']


## 2.2 Sampling Rate Check

`raw_timestamp` is in **milliseconds**, so the sampling rate is `1000 / median(dt_ms)`.  
This is diagnostic only — data is not modified here.

In [3]:
def check_sampling_rate(df, participant_id, time_col="raw_timestamp"):
    all_intervals = []
    for _, sub in df.groupby(["condition_number", "trial_number"], sort=True):
        vals = pd.to_numeric(sub[time_col], errors="coerce").sort_values().dropna().to_numpy()
        if len(vals) >= 2:
            all_intervals.append(np.diff(vals))
    if not all_intervals:
        print(f"  ⚠️  Participant {participant_id}: not enough data")
        return None
    dt_ms = np.concatenate(all_intervals)
    median_dt_ms = float(np.median(dt_ms))
    sampling_rate_hz = 1000.0 / median_dt_ms if median_dt_ms > 0 else float("nan")
    status = "✅" if sampling_rate_hz >= 80 else "⚠️"
    print(f"  {status}  Participant {participant_id}  —  "
          f"{sampling_rate_hz:.1f} Hz  ({median_dt_ms:.2f} ms/sample)")
    return {
        "participant_id": participant_id,
        "sampling_hz":    round(sampling_rate_hz, 2),
        "median_ms":      round(median_dt_ms, 3),
        "n_samples":      len(df),
    }


## 2.3 Relative Time Axis

Adds `time_ms`: milliseconds elapsed since the first sample of each Lego model.  
Since `raw_timestamp` is already in ms, no unit conversion is needed.

In [4]:
def add_time_per_model(df, time_col="raw_timestamp"):
    """
    Add `time_ms` per model_name, respecting experiment order.
    Sorting: condition_number -> trial_number -> time_col.
    Assumes time_col is in milliseconds.
    """
    df = df.copy()
    df = df.sort_values(["condition_number", "trial_number", time_col])
    t0 = df.groupby("model_name")[time_col].transform("first")
    df["time_ms"] = (df[time_col] - t0).astype(float)
    return df


## 2.4 Validity & Interpolation

**Validity rule (per tracker, per row):** invalid if `(pos_x, pos_y, pos_z) == (0, 0, 0)` (mistracking) or any value is non-finite.

Each tracker is processed independently:
1. Find contiguous runs of bad samples on that tracker.
2. Pad each run by 2 samples on both sides, then merge overlaps.
3. If the padded run is bracketed by valid samples on left and right, linearly interpolate `pos_x`, `pos_y`, `pos_z` against `time_ms`.
4. Output columns per tracker: `clean_<T>_pos_x/y/z`, `bad_sample_<T>`, `is_interpolated_<T>`.

Quaternion rotations (`rot_x/y/z/w`) are **not** interpolated (would require slerp).


In [5]:
def good_position(xyz):
    """True for valid samples: finite and not (0, 0, 0)."""
    xyz = np.asarray(xyz, dtype=float)
    finite  = np.isfinite(xyz).all(axis=1)
    is_zero = (xyz == 0).all(axis=1)
    return finite & ~is_zero


def find_runs(mask):
    """Return list of (start, end) inclusive index pairs for True-runs in mask."""
    mask = np.asarray(mask, dtype=bool)
    idx  = np.flatnonzero(mask)
    if len(idx) == 0:
        return []
    runs = []
    start = prev = idx[0]
    for k in idx[1:]:
        if k == prev + 1:
            prev = k
        else:
            runs.append((start, prev))
            start = prev = k
    runs.append((start, prev))
    return runs


def expand_runs(runs, n, pad=2):
    """Expand each run by `pad` samples on both sides, then merge overlaps."""
    if not runs:
        return []
    expanded = [(max(0, a - pad), min(n - 1, b + pad)) for a, b in runs]
    expanded.sort()
    merged = [expanded[0]]
    for a, b in expanded[1:]:
        pa, pb = merged[-1]
        if a <= pb + 1:
            merged[-1] = (pa, max(pb, b))
        else:
            merged.append((a, b))
    return merged


def interpolate_tracker_segment(sub, tracker, pad=2):
    """
    Interpolate one tracker's pos_x/y/z within a (condition, trial) segment.
    Adds: clean_<T>_pos_x/y/z, bad_sample_<T>, is_interpolated_<T>.
    """
    sub      = sub.sort_values("time_ms").copy()
    t        = sub["time_ms"].to_numpy(dtype=float)
    pos_cols = POS_COLS[tracker]
    xyz      = np.column_stack([
        pd.to_numeric(sub[c], errors="coerce").to_numpy(dtype=float)
        for c in pos_cols
    ])

    good     = good_position(xyz)
    bad_runs = find_runs(~good)
    n        = len(sub)
    clean    = xyz.copy()
    is_interp = np.zeros(n, dtype=bool)

    buffered = expand_runs(bad_runs, n=n, pad=pad)

    for a, b in buffered:
        clean[a:b + 1, :] = np.nan

    for a, b in buffered:
        left, right = a - 1, b + 1
        bracketed = (
            left >= 0 and right < n and
            good_position(xyz[[left]])[0] and
            good_position(xyz[[right]])[0]
        )
        if bracketed:
            for dim in range(3):
                clean[a:b + 1, dim] = np.interp(
                    t[a:b + 1],
                    [t[left], t[right]],
                    [xyz[left, dim], xyz[right, dim]],
                )
            is_interp[a:b + 1] = True

    sub[f"bad_sample_{tracker}"]    = ~good
    sub[f"is_interpolated_{tracker}"] = is_interp
    for j, c in enumerate(pos_cols):
        sub[f"clean_{c}"] = clean[:, j]

    return sub


def interpolate_all_segments(df, pad=2):
    """For each (condition, trial) group, interpolate every tracker independently."""
    out = []
    for (cond, trial), sub in df.groupby(["condition_number", "trial_number"], sort=True):
        sub = sub.sort_values("time_ms").copy()
        for T in TRACKERS:
            sub = interpolate_tracker_segment(sub, tracker=T, pad=pad)
        sub["segment_label"] = f"C{int(cond)}_T{int(trial)}"
        out.append(sub)
    return (
        pd.concat(out, axis=0)
        .sort_values(["condition_number", "trial_number", "time_ms"])
        .reset_index(drop=True)
    )


# 3. Full Pipeline

`process_participant` loads, cleans, and preprocesses all data for one participant.

In [6]:
def process_participant(participant_id):
    """Full preprocessing pipeline for one participant."""
    file_paths = get_participant_files(participant_id)
    if not file_paths:
        return pd.DataFrame(), {}

    # load & concatenate all condition files
    df = load_participant(file_paths, participant_id)

    # sampling rate check (diagnostic only)
    sr = check_sampling_rate(df, participant_id)


    # relative time axis
    df = add_time_per_model(df, time_col="raw_timestamp")

    # validity marking + interpolation
    df = interpolate_all_segments(df, pad=2)

    # sort
    df = (
        df
        .sort_values(["condition_number", "trial_number", "raw_timestamp"])
        .reset_index(drop=True)
    )

    summary = {
        "participant_id": participant_id,
        "n_files":        len(file_paths),
        "source_files":   [fp.name for fp in file_paths],
        "n_rows":         int(len(df)),
        "sampling_hz":    sr["sampling_hz"] if sr else np.nan,
    }

    return df, summary


# 4. Save

Run the full pipeline for every participant and save one CSV each to `data/body_tracking/processed/`.

In [7]:
participant_ids = get_participant_ids()
print(f"Processing {len(participant_ids)} participant(s): {participant_ids}\n")

summary_rows = []
failed       = []

pbar = tqdm(participant_ids, desc="participants", bar_format=B_FORMAT, dynamic_ncols=True)

for pid in pbar:
    pbar.set_postfix_str(f"current → {pid}")
    try:
        df_out, summary = process_participant(pid)

        if df_out.empty:
            failed.append(pid)
            continue

        out_path = OUTPUT_DIR / f"{pid}_cleaned_BT.csv"
        df_out.to_csv(out_path, index=False)
        summary_rows.append(summary)
        print(f"  ✅  {pid}  →  {out_path}  ({len(df_out):,} rows)")

    except Exception as exc:
        print(f"  ❌  {pid} failed: {exc}")
        failed.append(pid)

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("participant_id")
    .reset_index(drop=True)
)

print(f"\n✅  Saved {len(summary_rows)} CSV(s) → {OUTPUT_DIR}")
if failed:
    print(f"❌  Failed / skipped: {failed}")
print()
print(summary_df.to_string())


Processing 5 participant(s): ['001', '002', '003', '004', '005']



📄 0 of 5 participants processed:           
📄 0 of 5 participants processed:           
      0%  ⏱️ 00:00  ⏳ ?  ⚙️ ?it/s, current → 001

  ✅  Participant 001  —  90.9 Hz  (11.00 ms/sample)


📄 1 of 5 participants processed: ██        
📄 1 of 5 participants processed: ██        urrent → 001
     20%  ⏱️ 00:15  ⏳ 01:00  ⚙️ 15.22s/it, current → 002

  ✅  001  →  ../data/body_tracking/processed/001_cleaned_BT.csv  (368,021 rows)
  ✅  Participant 002  —  90.9 Hz  (11.00 ms/sample)


...

  ✅  003  →  ../data/body_tracking/processed/003_cleaned_BT.csv  (290,438 rows)
  ✅  Participant 004  —  90.9 Hz  (11.00 ms/sample)


📄 4 of 5 participants processed: ████████  
📄 4 of 5 participants processed: ████████  urrent → 004
     80%  ⏱️ 00:58  ⏳ 00:14  ⚙️ 14.80s/it, current → 005

  ✅  004  →  ../data/body_tracking/processed/004_cleaned_BT.csv  (413,560 rows)
  ✅  Participant 005  —  90.9 Hz  (11.00 ms/sample)


📄 5 of 5 participants processed: ██████████
📄 5 of 5 participants processed: ██████████urrent → 005
    100%  ⏱️ 01:15  ⏳ 00:00  ⚙️ 15.02s/it, current → 005

  ✅  005  →  ../data/body_tracking/processed/005_cleaned_BT.csv  (384,607 rows)

✅  Saved 5 CSV(s) → ../data/body_tracking/processed

  participant_id  n_files                                                                                                                                                  source_files  n_rows  sampling_hz
0            001        4  [001_BT_Data_Condition0_2026-05-06.csv, 001_BT_Data_Condition1_2026-05-06.csv, 001_BT_Data_Condition2_2026-05-06.csv, 001_BT_Data_Condition3_2026-05-06.csv]  368021        90.91
1            002        4  [002_BT_Data_Condition0_2026-05-06.csv, 002_BT_Data_Condition1_2026-05-06.csv, 002_BT_Data_Condition2_2026-05-06.csv, 002_BT_Data_Condition3_2026-05-06.csv]  340501        90.91
2            003        4  [003_BT_Data_Condition0_2026-05-06.csv, 003_BT_Data_Condition1_2026-05-06.csv, 003_BT_Data_Condition2_2026-05-06.csv, 003_BT_Data_Condition3_2026-05-06.csv]  290438        90.91
3            004        4  [004_BT_Data_Condit